In [1]:
# --- Reproducibility Seed Setup ---
import os
import random
import numpy as np
import torch

SEED = 3180

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Global random seed set:", SEED)

✅ Global random seed set: 3180


In [2]:
import sys, site, platform
print("PY:", platform.python_version())
print("EXE:", sys.executable)
print("USER_SITE:", site.getusersitepackages())


PY: 3.9.18
EXE: /common/software/install/manual/pytorch/2.1.2-pyclustertend/bin/python
USER_SITE: /users/4/volko028/.local/lib/python3.9/site-packages


In [3]:
# !python -m pip install torch transformers accelerate

In [4]:
# # Use the notebook's interpreter, not the system default
# !"{sys.executable}" -m pip install -U --no-cache-dir transformers==4.45.2 accelerate>=0.33

# # If you also need to ensure your CUDA torch is in THIS interpreter:
# !"{sys.executable}" -m pip install -U --no-cache-dir \
#   --index-url https://download.pytorch.org/whl/cu128 \
#   torch==2.9.0


In [5]:
import torch, transformers, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.1.2
transformers: 4.45.2
accelerate: 1.10.1


In [6]:
import os, torch, platform
print("python:", platform.python_version())
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("cuda.is_available():", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count(), "visible:", os.environ.get("CUDA_VISIBLE_DEVICES"))

python: 3.9.18
torch: 2.1.2 cuda: 11.8
cuda.is_available(): True
device_count: 1 visible: 0


In [7]:
# --- Imports ---
from sklearn.model_selection import train_test_split
import math, torch, pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import numpy as np
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import (
    precision_recall_curve, f1_score, precision_score, recall_score, roc_auc_score
)
from scipy.special import expit
import numpy as np


2026-03-08 15:54:20.138131: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-08 15:54:20.138244: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-08 15:54:20.138258: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-08 15:54:20.477060: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
# --- Config ---
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN    = 256
PER_DEVICE_BS = 2
GRAD_ACCUM    = 4
NUM_EPOCHS    = 3
EVAL_EVERY    = 2000
SAVE_EVERY    = 2000
STRIDE=32

In [15]:
# --- Load data ---
import pandas as pd

use_cols = ["notes", "outcome_hospitalization"]
train_df = pd.read_csv("mv_train.csv", usecols = use_cols)
val_df   = pd.read_csv("mv_val.csv", usecols = use_cols)
test_df  = pd.read_csv("mv_test.csv", usecols = use_cols)

In [16]:
# --- Tokenizer & collator ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

from transformers import DataCollatorWithPadding
import torch

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def tokenize_with_stride(text: str):
    text = text[:6000]
    return tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        padding=False
    )


In [17]:
# --- Streaming Dataset Loader ---
import math
import pandas as pd
import torch
import random
import numpy as np

class StreamingNotes(torch.utils.data.IterableDataset):
    def __init__(
        self,
        csv_path: str,
        dup_pos: int = 1,
        shuffle_within_chunk: bool = True,
        seed: int = 0,
        tokenizer=None,
        length_cache=None,
        max_len: int = 256,
        stride: int = 32,
        max_char: int = 6000,
        chunksize: int = 8192,
        text_col: str = "notes",
        label_col: str = "outcome_hospitalization",
    ):
        super().__init__()
        self.csv_path = csv_path
        self.dup_pos = max(1, int(dup_pos))
        self.shuffle_within_chunk = shuffle_within_chunk
        self.seed = seed

        self.tokenizer = tokenizer
        self.max_len = int(max_len)
        self.stride = int(stride)
        self.max_char = int(max_char)
        self.chunksize = int(chunksize)

        self.text_col = text_col
        self.label_col = label_col

        self._length = length_cache
        if self._length is None and self.tokenizer is not None:
            self._length = self._estimate_length()

    def __len__(self):
        if self._length is None:
            raise TypeError(
                "StreamingNotes has no length. Pass tokenizer=... so it can estimate __len__, "
                "or pass length_cache=<int>."
            )
        return int(self._length)

    def _tokenize_batch(self, texts):
        return self.tokenizer(
            texts,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
            stride=self.stride,
            return_overflowing_tokens=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            padding=False
        )

    def _estimate_length(self):
        total = 0
        usecols = [self.text_col, self.label_col]

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            texts = chunk[self.text_col].astype(str).str.slice(0, self.max_char).tolist()
            labels = chunk[self.label_col].astype(int).to_numpy()

            enc = self._tokenize_batch(texts)
            mapping = enc.get("overflow_to_sample_mapping", None)

            if mapping is None:
                for y in labels:
                    repeats = self.dup_pos if y == 1 else 1
                    total += repeats
            else:
                mapping = np.asarray(mapping)
                for j in range(len(texts)):
                    n_win = int(np.sum(mapping == j))
                    repeats = self.dup_pos if labels[j] == 1 else 1
                    total += repeats * n_win

        return int(total)

    def __iter__(self):
        rng = random.Random(self.seed)
        usecols = [self.text_col, self.label_col]

        global_note_id = 0

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            rows = list(chunk.iterrows())
            if self.shuffle_within_chunk:
                rng.shuffle(rows)

            for _, row in rows:
                y = int(row[self.label_col])
                text = str(row[self.text_col])[:self.max_char]

                enc = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=True,
                    max_length=self.max_len,
                    stride=self.stride,
                    return_overflowing_tokens=True,
                    return_attention_mask=True,
                    return_token_type_ids=True,
                    padding=False
                )

                n = len(enc["input_ids"])
                repeats = self.dup_pos if y == 1 else 1

                for _ in range(repeats):
                    for k in range(n):
                        item = {
                            "input_ids": enc["input_ids"][k],
                            "attention_mask": enc["attention_mask"][k],
                            "labels": y,
                            "note_id": global_note_id,
                        }
                        if "token_type_ids" in enc:
                            item["token_type_ids"] = enc["token_type_ids"][k]
                        yield item

                global_note_id += 1
                
                
                
train_ds = StreamingNotes(
    "mv_train.csv",
    dup_pos=1,
    shuffle_within_chunk=True,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

val_ds = StreamingNotes(
    "mv_val.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)



In [18]:
# --- Model ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: "NoAdmit", 1: "Admit"}
model.config.label2id = {"NoAdmit": 0, "Admit": 1}

/common/software/install/manual/pytorch/2.1.2-pyclustertend/lib/python3.9/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# --- Training Arguments ---
n_train = len(train_ds)
world_size = max(1, torch.cuda.device_count())
effective_bsz = PER_DEVICE_BS * GRAD_ACCUM * world_size
steps_per_epoch = math.ceil(n_train / effective_bsz)
MAX_STEPS = steps_per_epoch * NUM_EPOCHS
print(f"Training rows={n_train}  steps/epoch={steps_per_epoch}  max_steps={MAX_STEPS}")
assert MAX_STEPS > 0, "No training rows found. Check CSV path and columns."

training_args = TrainingArguments(
    output_dir="runs/clin-longformer",
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=-1,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    tf32=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    eval_accumulation_steps=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_strategy="steps",
    save_steps=SAVE_EVERY,
    load_best_model_at_end=True,
    metric_for_best_model="note_auroc",
    greater_is_better=True,
    save_total_limit=3,
    remove_unused_columns=False,
    include_inputs_for_metrics=True,
    report_to="none",
    gradient_checkpointing=True,
)

Training rows=130981  steps/epoch=16373  max_steps=49119


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
# --- Create Trainer ---
import numpy as np
import torch
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score

class NoteAwareTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        inputs.pop("note_id", None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = dict(inputs)
        note_ids = inputs.pop("note_id", None)

        loss, logits, labels = super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys
        )

        if note_ids is not None:
            if not isinstance(note_ids, torch.Tensor):
                note_ids = torch.as_tensor(note_ids, dtype=torch.long)
            note_ids = note_ids.view(-1, 1)
            return loss, (logits, note_ids), labels

        return loss, logits, labels

POOL = "mean"

def compute_metrics_note_level(eval_pred):
    preds_obj, labels = eval_pred.predictions, eval_pred.label_ids
    logits, note_ids = preds_obj
    if isinstance(logits, torch.Tensor): logits = logits.detach().cpu().numpy()
    if isinstance(note_ids, torch.Tensor): note_ids = note_ids.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor): labels = labels.detach().cpu().numpy()
    note_ids = note_ids.squeeze(-1)

    logits = np.asarray(logits)
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)
    p1 = probs[:, 1]

    # pool by note
    by_note_probs, by_note_label = {}, {}
    for nid, pr, y in zip(note_ids, p1, labels):
        nid = int(nid)
        by_note_probs.setdefault(nid, []).append(float(pr))
        by_note_label[nid] = int(y)

    if POOL == "max":
        note_probs = np.array([max(v) for nid, v in sorted(by_note_probs.items())])
    else:
        note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
    note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

    # rank metrics
    try:
        auroc = roc_auc_score(note_labels, note_probs)
    except Exception:
        auroc = float("nan")
    try:
        auprc = average_precision_score(note_labels, note_probs)
    except Exception:
        auprc = float("nan")

    # find best F1 threshold on this eval set
    precs, recs, ths = precision_recall_curve(note_labels, note_probs)
    f1s = 2 * precs * recs / (precs + recs + 1e-12)
    best_ix = int(np.nanargmax(f1s))
    best_th = ths[best_ix-1] if best_ix > 0 and best_ix <= len(ths) else 0.5
    target_p = 0.80
    ix_p = np.where(precs >= target_p)[0]

    # report metrics at best_th
    note_pred_best = (note_probs >= best_th).astype(int)
    acc = accuracy_score(note_labels, note_pred_best)
    prec, rec, f1, _ = precision_recall_fscore_support(note_labels, note_pred_best, average="binary", zero_division=0)
    
    print({
      "true_pos_rate": float(np.mean(note_labels)),
      "pred_pos_rate@0.5": float(np.mean((note_probs >= 0.5).astype(int))),
      "mean_prob_pos": float(np.mean(note_probs[note_labels==1])) if np.any(note_labels==1) else None,
      "mean_prob_neg": float(np.mean(note_probs[note_labels==0])) if np.any(note_labels==0) else None,
      "best_th": float(best_th),
    })

    return {
        "note_auroc": auroc,
        "note_auprc": auprc,
        "note_f1": f1,
        "note_precision": prec,
        "note_recall": rec,
        "note_acc": acc,
        "th_best_f1": float(best_th),
    }

In [22]:
# --- Model Training ---
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from transformers import TrainerCallback
import time

trainer = NoteAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_note_level,
    )
trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Note Auroc,Note Auprc,Note F1,Note Precision,Note Recall,Note Acc,Th Best F1
2000,0.615100,0.621566,0.690757,0.741332,0.754154,0.631583,0.935758,0.637929,0.360180
4000,0.609300,0.615271,0.704629,0.751381,0.758203,0.650519,0.908611,0.656069,0.307816
6000,0.624900,0.609449,0.710480,0.754415,0.760738,0.650132,0.916693,0.657791,0.377885
8000,0.626400,0.614537,0.710764,0.750708,0.760304,0.654479,0.906953,0.660620,0.449242
10000,0.620900,0.606180,0.714054,0.759969,0.760012,0.656878,0.901565,0.662096,0.390069
12000,0.593600,0.612208,0.714670,0.757371,0.761346,0.663257,0.893483,0.667569,0.420198
14000,0.608300,0.619274,0.717372,0.761333,0.761976,0.662960,0.895762,0.667876,0.504822
16000,0.628300,0.607631,0.717262,0.762293,0.760182,0.640862,0.934100,0.650228,0.413744
18000,0.575700,0.607642,0.717875,0.762093,0.761047,0.647183,0.923531,0.655823,0.335460
20000,0.600600,0.605930,0.719625,0.762634,0.764252,0.659204,0.909129,0.667138,0.408837


{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.6701512729061616, 'mean_prob_pos': 0.6551081464211813, 'mean_prob_neg': 0.5389591874251872, 'best_th': 0.3601795434951782}
{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.65465502398229, 'mean_prob_pos': 0.6436533659045011, 'mean_prob_neg': 0.49507110051651837, 'best_th': 0.30781620740890503}
{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.67119665477801, 'mean_prob_pos': 0.6300192280079884, 'mean_prob_neg': 0.4997156275448397, 'best_th': 0.37788498401641846}
{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.771184356167753, 'mean_prob_pos': 0.6969740000314978, 'mean_prob_neg': 0.5706051184097481, 'best_th': 0.44924160838127136}
{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.6883532160865822, 'mean_prob_pos': 0.6366426491367423, 'mean_prob_neg': 0.5026088303845078, 'best_th': 0.3900691866874695}
{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.7184848112163326, 'm

TrainOutput(global_step=49116, training_loss=0.5913800798688439, metrics={'train_runtime': 16853.8651, 'train_samples_per_second': 23.315, 'train_steps_per_second': 2.914, 'total_flos': 4.280858804031648e+16, 'train_loss': 0.5913800798688439, 'epoch': 2.999862576537234})

In [23]:
test_ds = StreamingNotes(
    "mv_test.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

In [34]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

# use test dataset
eval_loader = trainer.get_eval_dataloader(test_ds)

# number of test examples
n_test = len(test_ds)

world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_test / bsz))
print(f"n_test={n_test}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

# concatenate predictions
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)

metrics = compute_metrics_note_level(eval_pred)
print("\nTest metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))


n_test=16382, bsz=2, total_steps=8191
Predicting: 100%|##########| 8191/8191 [01:33<00:00, 87.82it/s]
{'true_pos_rate': 0.5947607920305006, 'pred_pos_rate@0.5': 0.7336121018324929, 'mean_prob_pos': 0.6951741680419696, 'mean_prob_neg': 0.5452785410156561, 'best_th': 0.41599568724632263}

Test metrics: {'note_auroc': 0.7203823184989664, 'note_auprc': 0.7684429182527488, 'note_f1': 0.7660278745644599, 'note_precision': 0.661800120409392, 'note_recall': 0.9092224979321754, 'note_acc': 0.66965932849588, 'th_best_f1': 0.41599568724632263}

Confusion matrix:
[[2096 4494]
 [ 878 8794]]

Classification report:
              precision    recall  f1-score   support

           0     0.7048    0.3181    0.4383      6590
           1     0.6618    0.9092    0.7660      9672

    accuracy                         0.6697     16262
   macro avg     0.6833    0.6136    0.6022     16262
weighted avg     0.6792    0.6697    0.6332     16262



In [35]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       1         0.907967
1          1       1         0.625266
2          2       0         0.738681
3          3       1         0.490144
4          4       1         0.839175


In [36]:
pred_df

,sample_id,y_true,pred_prob_notes
0,0,1,0.907967
1,1,1,0.625266
2,2,0,0.738681
3,3,1,0.490144
4,4,1,0.839175
...,...,...,...
16257,16257,0,0.772949
16258,16258,1,0.858452
16259,16259,0,0.360250
16260,16260,1,0.709560


In [37]:
from sklearn.metrics import roc_auc_score

auroc = roc_auc_score(note_labels, note_probs)
print("Note-level AUROC:", auroc)

Note-level AUROC: 0.7203823184989664


In [38]:
pred_df.to_csv("notes_test_predictions.csv", index=False)

In [39]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

# use validation dataset
eval_loader = trainer.get_eval_dataloader(val_ds)

# number of validation examples
n_test = len(val_ds)

world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_test / bsz))
print(f"n_test={n_test}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

# concatenate predictions
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)

metrics = compute_metrics_note_level(eval_pred)
print("\nTest metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))

n_test=16358, bsz=2, total_steps=8179
Predicting: 100%|##########| 8179/8179 [01:31<00:00, 89.13it/s]
{'true_pos_rate': 0.5934694379535113, 'pred_pos_rate@0.5': 0.7342270323453449, 'mean_prob_pos': 0.6954525550428838, 'mean_prob_neg': 0.5446751168545156, 'best_th': 0.4088371992111206}

Test metrics: {'note_auroc': 0.7196246805055975, 'note_auprc': 0.7626344877608326, 'note_f1': 0.7642524280301382, 'note_precision': 0.6592036063110444, 'note_recall': 0.909128587711118, 'note_acc': 0.6671381133931865, 'th_best_f1': 0.4088371992111206}

Confusion matrix:
[[2075 4536]
 [ 877 8774]]

Classification report:
              precision    recall  f1-score   support

           0     0.7029    0.3139    0.4340      6611
           1     0.6592    0.9091    0.7643      9651

    accuracy                         0.6671     16262
   macro avg     0.6811    0.6115    0.5991     16262
weighted avg     0.6770    0.6671    0.6300     16262



In [40]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       1         0.828592
1          1       1         0.892656
2          2       0         0.504638
3          3       1         0.652006
4          4       1         0.412353


In [41]:
pred_df.to_csv("notes_val_predictions.csv", index=False)